In [ ]:
# # Description
# 
# Prove that the Noise Reduction model works on a 2D polinomial.

# Add libs folder path to system variables to make the importation of local libraries easier.
# __________________________________________________________________________________________________
import os
import sys


CURRENT_DIR = os.getcwd()
FOLDERS = CURRENT_DIR.split(os.sep)
TESIS_FOLDER_INDEX = FOLDERS.index('S-noise-gradient')
CURRENT_DIR = os.sep.join(FOLDERS[:TESIS_FOLDER_INDEX+1])
LIBS_PATH = os.path.join(CURRENT_DIR, 'src', 'libs')
assert os.path.exists(LIBS_PATH)
sys.path.append(LIBS_PATH)


# Import external and local libraries
# __________________________________________________________________________________________________
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import json
from typing import List

# Locals
from utils import *
from dataset import DFDataset, TensorDataset
from models import *
import config as cfg
from dlnr import DLNoiseReduction


# Global / Path variables
# __________________________________________________________________________________________________
DATA_PATH = os.path.join(CURRENT_DIR, 'data')
CHECKPOINT_PATH = os.path.join(CURRENT_DIR, 'checkpoints')
CONFIG_PATH = os.path.join(CURRENT_DIR, 'config')


# Make sure that the GPU is being used
# __________________________________________________________________________________________________
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda"

In [ ]:
device = 'cpu'

In [ ]:
def add_gaussian_noise(df:pd.DataFrame, columns:List[str], mean:float = 0.0, std:float = 0.1):
    """
    Adds Gaussian noise to specified columns in a DataFrame.

    Args:
        df (pandas.DataFrame): The DataFrame to which noise will be added.
        columns (list): A list of column names in the DataFrame to which noise will be added.
        mean (int, optional): The mean of the Gaussian distribution. Defaults to 0.
        std (float, optional): The standard deviation of the Gaussian distribution. Defaults to 0.1.

    Returns:
        pandas.DataFrame: The DataFrame with added Gaussian noise to specified columns.
    """
    for col in columns:
        df[col] = df[col] + np.random.normal(loc=mean, scale=std, size=len(df))
    return df

def polinomial_function(x:float):
    """
    Function that takes in a value x and returns its polinomial value.

    Args:
        x (float): value to be transformed.

    Returns:
        float: result of the polinomial function.
    """

    return x**4 -x**3 -20*(x**2) -20*x +6

In [ ]:
data = np.arange(-5, 5, 0.001)
data_df = pd.DataFrame(data, columns=['x'])
data_df['y'] = polinomial_function(data_df['x'].values)

scaler = MinMaxScaler()
data_df = pd.DataFrame(scaler.fit_transform(data_df), columns=['x', 'y'])

In [ ]:
data_df.shape

In [ ]:
plt.plot(data_df['x'], data_df['y'])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data_df['x'].values, data_df['y'].values, test_size=0.2, random_state=42
)

# No noise XAI Benchmarck

In [ ]:
model_params = {
    'ridge': {"alpha": 1.0},
    'pls': {"n_components": 1},
    'tree': {"max_depth": 5},
    'svm': {"dual": 'auto'},
    'knn': {"n_neighbors": 5, "weights": 'uniform', "algorithm": 'auto', "leaf_size": 30, "p": 2, "n_jobs": None},
}
xai_benchmark = XAI_benchmark(is_ts = False, model_params = model_params)

In [ ]:
# xai_benchmark.fit(data_df[['x']], data_df['y'].values)
xai_benchmark.fit(X_train.reshape(-1,1), y_train)

In [ ]:
# xai_benchmark.predict(data_df[['x']], data_df['y'].values, get_metrics=True)
xai_benchmark.predict(X_test.reshape(-1,1), y_test, get_metrics=True)

Add goussian noise to the dataframe

In [ ]:
sigma = 0.05
df_noisy = add_gaussian_noise(
        df=data_df.copy(),
        columns=[x for x in data_df.columns],
        mean=0,
        std=sigma
)

In [ ]:
X_train_noisy, X_test_noisy, y_train_noisy, y_test_noisy = train_test_split(
    df_noisy['x'].values, df_noisy['y'].values, test_size=0.2, random_state=42
)

In [ ]:
plt.scatter(X_train_noisy, y_train_noisy, s=1, c='r', alpha=0.5)

# Noisy XAI Benchmarck

In [ ]:
xai_benchmark.fit(X_train_noisy.reshape(-1,1), y_train_noisy)

In [ ]:
xai_benchmark.predict(X_test_noisy.reshape(-1,1), y_test_noisy, get_metrics=True)

Create Datasets for DataLoaders

In [ ]:
train_dataset = TensorDataset(X_train_noisy, y_train_noisy)
test_dataset = TensorDataset(X_test_noisy, y_test_noisy)

# DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
denoiser = DenoisingAutoencoder(
    input_dim=1,
    latent_dim=32,
).to(device)

# denoiser_v2 = DenoisingAutoencoder_v2(
#     input_dim=1,
#     hidden_dims=[256, 128, 64, 32],
#     encoding_dim=8
# ).to(device)

# n_layers = 3
# hidden_layers = [(1, 64), (64, 32), (32, 1)]
# dropout_layers = [0.1, 0.2, 0.0]
# activation_func_layers = [nn.ReLU(), nn.ReLU(), nn.Identity()]
# want_dropout = [False, True, False]
# want_linear = [True, True, True]
# want_activation = [True, True, False]

# dense_nn = GridFullyDenseNN(
#     n_layers, hidden_layers, dropout_layers, activation_func_layers,
#     want_dropout, want_linear, want_activation
# ).to(device)

In [ ]:
# Set model parameters and create the model Trainer object
lr = 0.001
criterion = nn.MSELoss()
optimizer = optim.Adam(denoiser.parameters(), lr=lr)

# Define the trainer
trainer_basic = Trainer(
    model=denoiser,
    train_generator=train_loader,
    val_generator=test_loader,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    epoch_scheduler=None,
    batch_scheduler=None,
    patience=15,
    epochs=1000,
    checkpoints_path=os.path.join(
        CHECKPOINT_PATH, f'denoising_autoencoder_sigma{sigma}'
    ),
)

In [ ]:
# denoiser = torch.load(os.path.join(CHECKPOINT_PATH, 'denoising_autoencoder.pth'))

In [ ]:
# Train the model
model, train_losses, val_losses, best_train_loss, best_val_loss = trainer_basic.fit(verbose=True)

In [ ]:
y_pred_noisy = trainer_basic.eval_dataloader(test_loader)
flattened_tensor = torch.cat(y_pred_noisy).view(-1)
len(flattened_tensor)

In [ ]:
# Show the metrics
gt_values = y_test
predicted_values = flattened_tensor.cpu().detach().numpy()

mae = mean_absolute_error(gt_values, predicted_values)
mse = mean_squared_error(gt_values, predicted_values)
rmse = np.sqrt(mse)
# mape = np.mean(np.abs(gt_values - predicted_values) / gt_values) * 100
r_squared = r2_score(gt_values, predicted_values)
nn_benchmark = {
    'nn': {
        "MSE: ": mse,
        "RMSE: ": rmse,
        "MAE: ": mae,
        # "MAPE: ": mape,
        "R2: ": r_squared
    }
}
nn_benchmark

# Denoise the entire noisy dataframe

In [ ]:
complete_dataset = TensorDataset(df_noisy['x'], df_noisy['y'])
complete_dataloader = DataLoader(complete_dataset, batch_size=batch_size, shuffle=False)
y_denoised = trainer_basic.eval_dataloader(complete_dataloader)
y_denoised = torch.cat(y_denoised).view(-1)
len(y_denoised)

In [ ]:
df_denoised = pd.DataFrame({'x': df_noisy['x'], 'y': y_denoised.cpu().detach().numpy()})

In [ ]:
X_train_denoised, X_test_denoised, y_train_denoised, y_test_denoised = train_test_split(
    df_denoised['x'].values, df_denoised['y'].values, test_size=0.2, random_state=42
)

In [ ]:
xai_benchmark.fit(X_train_denoised.reshape(-1,1), y_train_denoised)
xai_benchmark.predict(X_test_denoised.reshape(-1,1), y_test_denoised, get_metrics=True)

In [ ]:
xai_benchmark.predict(data_df[['x']], data_df[['y']].values, get_metrics=True)